# Get sequences features from Ensembl / Uniprot

In [2]:
import polars as pl
import pandas as pd
import os
from biomart import BiomartServer
from Bio.Seq import Seq
import re

In [2]:
outputs_dir = "../3_outputs"

## Convert sequences

In [10]:
metadata = pl.DataFrame(pd.read_csv(f"{outputs_dir}/varchamp_consolidated_alleles_clinvar_gnomad_sequence_confirmed.tsv", sep="\t"))
priority_col = ["symbol", "nt_change", "ensembl_gene_id", "sequence"]
metadata = metadata.select(priority_col).group_by("symbol").first()

sequences = metadata.select("sequence").to_series().to_list()
nt_changes = metadata.select("nt_change").to_series().to_list()

In [11]:
# Get wt nucleotide sequences and convert to aa sequences
wt_nt_sequences = []
wt_aa_sequences = []
for seq, nt in zip(sequences, nt_changes):
    pattern = r'(\d+)([A-Za-z])>([A-Za-z])'
    match = re.search(pattern, nt)
    nt_index = int(match.group(1)) - 1
    wt_nt = match.group(2)
    var_nt = match.group(3)

    seq_list = list(seq)

    if seq_list[nt_index] == var_nt:
        seq_list[nt_index] = wt_nt
        nt_seq = ''.join(seq_list)
        aa_seq = Seq(nt_seq).translate()

        wt_nt_sequences.append(nt_seq)
        wt_aa_sequences.append(str(aa_seq))
    else:
        wt_nt_sequences.append(None)
        wt_aa_sequences.append(None)

metadata = metadata.drop("sequence").with_columns(
    pl.Series(name="aa_sequence", values=wt_aa_sequences),
    pl.Series(name="nt_sequence", values=wt_nt_sequences),
)

In [ ]:
ensembl_gene_ids = metadata.select('ensembl_gene_id').to_series().unique().to_list()
server = BiomartServer("http://www.ensembl.org/biomart/martservice")
mart = server.datasets['hsapiens_gene_ensembl']

attributes = [
    'ensembl_gene_id',
    'ensembl_peptide_id_version',
    'peptide'
]


protein_ids = list(metadata["ensembl_gene_id"].unique())
batch_size = 100  
results = []

# Query BioMart in batches
for i in range(0, len(protein_ids), batch_size):
    batch = protein_ids[i : i + batch_size]
    
    filters = {
        'ensembl_gene_id': batch
    }
    
    try:
        response = mart.search({"attributes": attributes, "filters": filters})
        data = [line.decode("utf-8").split("\t") for line in response.iter_lines()]
        df_batch = pl.DataFrame(pd.DataFrame(data))
        df_batch.columns = ["amino_acid_sequence", "ensembl_gene_id", "ensembl_peptide_id_version"]
        results.append(df_batch)
    except Exception as e:
        print(f"Batch {i//batch_size + 1} failed: {e}")

all_aa_seqs = pl.concat(results)

In [ ]:
all_aa_seqs = all_aa_seqs.with_columns(
    pl.col("amino_acid_sequence").str.replace_all(r"\*", "").alias("aa_seq_cleaned")
)
all_aa_seqs.write_csv(f"{outputs_dir}/sequence_feature_annotations/1_all_wt_peptide_sequences.csv")

In [ ]:
# Need to find the perfect matches, then merge with our data
pm_aa_seqs = all_aa_seqs.filter(pl.col("aa_seq_cleaned").is_in(wt_aa_sequences))
pm_aa_seqs.write_csv(f"{outputs_dir}/sequence_feature_annotations/2_perfect_match_peptide_sequences.csv")

## Get sequence features

In [ ]:
# Get sequence features
server = BiomartServer("http://www.ensembl.org/biomart/martservice")
mart = server.datasets['hsapiens_gene_ensembl']

attributes = [
    "ensembl_peptide_id_version", "external_gene_name", "signalp", 
    "signalp_start", "signalp_end", "tmhmm", "tmhmm_start", 
    "tmhmm_end", "ncoils", "ncoils_start", "ncoils_end", "seg", 
    "seg_start", "seg_end", "pfam", "pfam_start", "pfam_end"
]

protein_ids = list(pm_aa_seqs["ensembl_peptide_id"].unique())
batch_size = 100  
results = []

# Query BioMart in batches
for i in range(0, len(protein_ids), batch_size):
    batch = protein_ids[i : i + batch_size]
    
    filters = {"ensembl_peptide_id_version": batch}
    
    try:
        response = mart.search({"attributes": attributes, "filters": filters})
        data = [line.decode("utf-8").split("\t") for line in response.iter_lines()]
        df_batch = pl.DataFrame(pd.DataFrame(data))
        results.append(df_batch)
        print(f"Batch {i//batch_size + 1} success!")
    except Exception as e:
        print(f"Batch {i//batch_size + 1} failed: {e}")
        

seq_features = pl.concat(results)
seq_features.columns = attributes
seq_features = seq_features.rename({"ensembl_peptide_id_version": "ensembl_peptide_id"})

Batch 1 success!
Batch 2 success!
Batch 3 success!
Batch 4 success!
Batch 5 success!
Batch 6 success!
Batch 7 success!
Batch 8 success!
Batch 9 success!
Batch 10 success!
Batch 11 success!
Batch 12 success!
Batch 13 success!
Batch 14 success!
Batch 15 success!
Batch 16 success!
Batch 17 success!
Batch 18 success!
Batch 19 success!
Batch 20 success!
Batch 21 success!
Batch 22 success!


In [ ]:
# there are multiple perfect matches with the same peptide ID - take the first and merge with seq features
pm_aa_seqs_unique = pm_aa_seqs.group_by("ensembl_gene_id").first()
ann_aa_seqs = pm_aa_seqs_unique.join(seq_features, on="ensembl_peptide_id")

In [41]:
# Convert to long format
common_ids = ['ensembl_gene_id', 'amino_acid_sequence', 'ensembl_peptide_id', 'aa_seq_cleaned', 'external_gene_name']
feat_types = ['signalp', 'tmhmm', 'ncoils', 'seg', 'pfam']

feat_df = []

for ft in feat_types:
    sel_feats = [ft, ft+"_start", ft+"_end"]
    df = ann_aa_seqs.select(common_ids + sel_feats).unique()
    df.columns = common_ids + ["feat_type", "aa_start", "aa_end"]

    feat_df.append(df)

seq_feats_long = pl.concat(feat_df).unique()
seq_feats_long = seq_feats_long.filter(pl.col("aa_start") != "")

In [47]:
# check to make sure our coordinates are mapping to our aa sequence
seq_feats_long = seq_feats_long.with_columns(
    pl.col("aa_seq_cleaned").str.len_chars().alias("num_aa"),
    pl.col("aa_start").cast(pl.Int64).alias("aa_start"),
    pl.col("aa_end").cast(pl.Int64).alias("aa_end"),
)

seq_feats_long = seq_feats_long.with_columns(
    (pl.col("aa_end") <= pl.col("num_aa")).alias("coordinates_qaqc"),
)

print(f"This number should be zero: {seq_feats_long.filter(~pl.col('coordinates_qaqc')).shape[0]}")

This number should be zero: 0


In [48]:
# Now, annotate our measured alleles
# process metadata
metadata = pl.DataFrame(pd.read_csv(f"{outputs_dir}/varchamp_consolidated_alleles_clinvar_gnomad_sequence_confirmed.tsv", sep="\t"))
mut_cols = ["mutation_id_rc4", "mutation_id_cegs2", "mutation_id_edgotyping3"]
id_vars = [col for col in metadata.columns if col not in mut_cols]
metadata = metadata.unpivot(index=id_vars, 
                            on=mut_cols,
                            variable_name="mutation_collection",
                            value_name="mut_id")

priority_col = ["orf_id", "mut_id", "symbol", "aa_change", "nt_change", "ensembl_gene_id", "clinvar_clnsig_clean", "gnomad_af", "StarStatus"]
metadata = metadata.select(priority_col).filter(~pl.col("mut_id").is_null()).with_columns(
    pl.col("mut_id").cast(pl.Int64).alias("mut_id"),
    pl.col("orf_id").cast(pl.Int64).alias("orf_id"),
    pl.concat_str([pl.col("symbol"), pl.col("aa_change")], separator="_").alias("allele_0")
)

In [56]:
allele_df = metadata.select(["ensembl_gene_id", "aa_change", "allele_0"]).unique().with_columns(
    pl.col("aa_change").str.replace_all(r"[^\d]", "").cast(pl.Int64).alias("aa_position")
)
alleles = allele_df.select("allele_0").to_series().unique().to_list()

In [61]:
allele_annotations = []
for allele in alleles:
    allele_temp = allele_df.filter(pl.col("allele_0") == allele)

    ens_gene = allele_temp.select("ensembl_gene_id").item()
    aa_pos = allele_temp.select("aa_position").item()

    seq_feat_temp = (seq_feats_long
                        .filter(pl.col("ensembl_gene_id") == ens_gene)
                        .filter(pl.col("aa_start") <= aa_pos)
                        .filter(pl.col("aa_end") >= aa_pos)
                        .with_columns(pl.lit(allele).alias("allele_0"))
                    ).select(["allele_0", "feat_type"])

    if seq_feat_temp.shape[0] > 0:
        allele_annotations.append(seq_feat_temp)

allele_annotations = pl.concat(allele_annotations)

In [63]:
allele_annotations.write_csv(f"{outputs_dir}/sequence_feature_annotations/3_allele_annotations.csv")